## Task 1

In [1]:
import pandas as pd 
import numpy as np 

cars_df = pd.read_csv('cars.csv')

In [2]:
cars_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    398 non-null    object 
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model year    398 non-null    int64  
 7   origin        398 non-null    int64  
 8   car name      398 non-null    object 
dtypes: float64(3), int64(4), object(2)
memory usage: 28.1+ KB


Horsepower is an object but should be numeric.  
Origin is int, but should be categorical.

## Task 2 Clean the Data

In [3]:
cars_df.isnull().sum()

mpg             0
cylinders       0
displacement    0
horsepower      0
weight          0
acceleration    0
model year      0
origin          0
car name        0
dtype: int64

In [4]:
X= cars_df.loc[:, 'cylinders':'origin']
y= cars_df['mpg']

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size= 0.2, random_state=42)

In [6]:
X_train['horsepower'] = pd.to_numeric(X_train['horsepower'], errors='coerce')
X_test['horsepower'] = pd.to_numeric(X_test['horsepower'], errors='coerce')

In [7]:
hp_med = X_train['horsepower'].median()
X_train['horsepower']= X_train['horsepower'].fillna(hp_med)
X_test['horsepower']= X_test['horsepower'].fillna(hp_med)

## Task 3 Decide What Goes In and What Stays Out

We split prior to cleaning the horsepower column above.  
We cannot use car name because they are individual string values and would not serve well as categorical or numeric columns  

## Task 4 When Should you split?

We split our data prior to cleaning the horsepower column to avoid data leakage when filling with the median value. If we had used the entire data set median, this would provide information about the data in the test set.  
The same concept would apply to scaling.  


## Task 5 Encode the Categorical Column

In [8]:
categorical_cols = ['origin', 'model year', 'cylinders']
numeric_cols = ['displacement']


In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

ohe= OneHotEncoder(sparse_output=False, drop='first') # This satisfies question 6
origin_array= ohe.fit_transform(X_train[['origin']])
# get column names from sklearn
origin_cols= ohe.get_feature_names_out(['origin'])
# build df from array
df_origin = pd.DataFrame(origin_array, columns=origin_cols, dtype=int)
df1=pd.concat([X_train.drop(columns=['origin']).reset_index(), df_origin], axis=1)
print(df1.to_string())



     index  cylinders  displacement  horsepower  weight  acceleration  model year  origin_2  origin_3
0        3          8         304.0       150.0    3433          12.0          70         0         0
1       18          4          97.0        88.0    2130          14.5          70         0         1
2      376          4          91.0        68.0    2025          18.2          82         0         1
3      248          4          91.0        60.0    1800          16.4          78         0         1
4      177          4         115.0        95.0    2694          15.0          75         1         0
5       63          8         400.0       175.0    4385          12.0          72         0         0
6      246          4          78.0        52.0    1985          19.4          78         0         1
7      352          4          98.0        65.0    2380          20.7          81         0         0
8      108          4          97.0        88.0    2279          19.0          73 

In [ ]:
# The following blocks split...for learning!
ohe= OneHotEncoder(sparse_output=False, drop='first')
origin_array= ohe.fit_transform(X_test[['origin']])
# get column names from sklearn
origin_array


In [17]:
origin_cols= ohe.get_feature_names_out(['origin'])
# build df from array
origin_cols


array(['origin_2', 'origin_3'], dtype=object)

In [18]:
df_origin = pd.DataFrame(origin_array, columns=origin_cols, dtype=int)
df_origin


,origin_2,origin_3
0,0,1
1,0,0
2,0,0
3,0,0
4,0,0
...,...,...
75,0,0
76,0,0
77,0,0
78,1,0


In [20]:
X_test.drop(columns=['origin']).reset_index()

,index,cylinders,displacement,horsepower,weight,acceleration,model year
0,198,4,91.0,53.0,1795,17.4,76
1,396,4,120.0,79.0,2625,18.6,82
2,33,6,232.0,100.0,2634,13.0,71
3,208,8,318.0,150.0,3940,13.2,76
4,93,8,318.0,150.0,4237,14.5,73
...,...,...,...,...,...,...,...
75,249,8,260.0,110.0,3365,15.5,78
76,225,6,250.0,110.0,3520,16.4,77
77,367,4,112.0,88.0,2605,19.6,82
78,175,4,90.0,70.0,1937,14.0,75


In [21]:
df=pd.concat([X_test.drop(columns=['origin']).reset_index(), df_origin], axis=1)
print(df)

    index  cylinders  displacement  horsepower  weight  acceleration  \
0     198          4          91.0        53.0    1795          17.4   
1     396          4         120.0        79.0    2625          18.6   
2      33          6         232.0       100.0    2634          13.0   
3     208          8         318.0       150.0    3940          13.2   
4      93          8         318.0       150.0    4237          14.5   
..    ...        ...           ...         ...     ...           ...   
75    249          8         260.0       110.0    3365          15.5   
76    225          6         250.0       110.0    3520          16.4   
77    367          4         112.0        88.0    2605          19.6   
78    175          4          90.0        70.0    1937          14.0   
79    285          8         305.0       130.0    3840          15.4   

    model year  origin_2  origin_3  
0           76         0         1  
1           82         0         0  
2           71         0

In the above code, we added drop='first' to drop origin 1 to avoid multicollinearity. It does not matter because the third answer is always implied by the other two responses.  